In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [3]:
!pip install deepspeed

In [4]:
ds_config = '''
{
  "train_batch_size": 64,
  "train_micro_batch_size_per_gpu": 32,
  "gradient_accumulation_steps": 1,

  "optimizer": {
    "type": "AdamW",
    "params": {
      "lr": 3e-4,
      "betas": [0.9, 0.999],
      "eps": 1e-8,
      "weight_decay": 0.01
    }
  },

  "scheduler": {
    "type": "WarmupLR",
    "params": {
      "warmup_min_lr": 0,
      "warmup_max_lr": 3e-4,
      "warmup_num_steps": 100
    }
  },

  "bf16": {
    "enabled": true
  },

  "zero_optimization": {
    "stage": 2,
    "allgather_partitions": true,
    "allgather_bucket_size": 2e8,
    "overlap_comm": true,
    "reduce_scatter": true,
    "reduce_bucket_size": 2e8,
    "contiguous_gradients": true
  },

  "gradient_clipping": 1.0,

  "wall_clock_breakdown": false
}
'''

with open("/kaggle/working/ds_config.json", "w") as f:
    f.write(ds_config)

print("Config written!")

Config written!


In [6]:
script = '''
import torch
import torch.nn as nn
import deepspeed
from torch.utils.data import DataLoader, TensorDataset


class TinyTransformer(nn.Module):
    def __init__(self, vocab_size=1000, hidden=256, num_layers=4, num_heads=4):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, hidden)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=hidden, nhead=num_heads,
            dim_feedforward=hidden * 4,
            batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.head = nn.Linear(hidden, vocab_size)

    def forward(self, x):
        x = self.embed(x)
        x = self.transformer(x)
        return self.head(x)


def make_dataset(num_samples=1024, seq_len=64, vocab_size=1000):
    x = torch.randint(0, vocab_size, (num_samples, seq_len))
    y = torch.randint(0, vocab_size, (num_samples, seq_len))
    return TensorDataset(x, y)


def main():
    model   = TinyTransformer()
    dataset = make_dataset()

    # ── deepspeed.initialize replaces: DDP() wrap + optimizer creation ──────
    # It reads ds_config.json and handles:
    #   - wrapping the model for distributed training
    #   - creating the optimizer (AdamW, per the config)
    #   - creating the LR scheduler (WarmupLR, per the config)
    #   - setting up ZeRO-2 sharding
    model_engine, optimizer, _, _ = deepspeed.initialize(
        model=model,
        model_parameters=model.parameters(),
        config="/kaggle/working/ds_config.json"
    )

    # model_engine.local_rank tells you which GPU this process owns
    rank = model_engine.local_rank

    def log(msg):
        if rank == 0:
            print(msg, flush=True)

    log(f"World size: {model_engine.world_size}")
    log(f"ZeRO stage: {model_engine.zero_optimization_stage()}")
    log("")

    # DataLoader — DeepSpeed does NOT auto-create a DistributedSampler for you
    # unlike DDP's typical setup, so we build it manually
    from torch.utils.data.distributed import DistributedSampler
    sampler = DistributedSampler(
        dataset,
        num_replicas=model_engine.world_size,
        rank=model_engine.global_rank,
        shuffle=True
    )
    loader = DataLoader(
        dataset,
        batch_size=model_engine.train_micro_batch_size_per_gpu(),
        sampler=sampler
    )

    criterion = nn.CrossEntropyLoss()

    epochs = 5
    for epoch in range(epochs):
        sampler.set_epoch(epoch)
        total_loss  = 0.0
        num_batches = 0

        for x, y in loader:
            # model_engine.device is the GPU this process owns
            x = x.to(model_engine.device)
            y = y.to(model_engine.device)

            logits = model_engine(x)
            loss   = criterion(logits.view(-1, 1000), y.view(-1))

            # ── These 3 lines replace DDP's loss.backward() + clip + step ───
            # backward(): triggers ZeRO's ReduceScatter (sharded, not full AllReduce)
            # step(): applies the optimizer update using each GPU's shard
            #         then broadcasts the updated params (ZeRO-2 behavior)
            # Gradient clipping is handled automatically per ds_config
            model_engine.backward(loss)
            model_engine.step()

            total_loss  += loss.item()
            num_batches += 1

        log(f"Epoch {epoch+1}/{epochs}  |  loss: {total_loss/num_batches:.4f}")

    # ── Memory check — same lesson as before, measure inside the process ────
    peak_mem = torch.cuda.max_memory_allocated(model_engine.local_rank) / 1e9
    print(f"[Rank {rank}] peak memory: {peak_mem:.3f} GB", flush=True)

    # ── Save checkpoint — DeepSpeed has its own checkpoint format ───────────
    # This saves sharded state automatically reassembled across GPUs
    model_engine.save_checkpoint("/kaggle/working/ds_checkpoint")
    log("Checkpoint saved to /kaggle/working/ds_checkpoint")


if __name__ == "__main__":
    main()
'''

with open("/kaggle/working/train_deepspeed.py", "w") as f:
    f.write(script)

print("Training script written!")

Training script written!


In [8]:
!deepspeed --num_gpus=2 /kaggle/working/train_deepspeed.py

[2026-06-30 21:55:12,821] [WARNING] [runner.py:232:fetch_hostfile] Unable to find hostfile, will proceed with training with local resources only.
[2026-06-30 21:55:12,822] [INFO] [runner.py:630:main] cmd = /usr/bin/python3 -u -m deepspeed.launcher.launch --world_info=eyJsb2NhbGhvc3QiOiBbMCwgMV19 --master_addr=127.0.0.1 --master_port=29500 --enable_each_rank_log=None --log_level=info /kaggle/working/train_deepspeed.py
[2026-06-30 21:55:22,739] [INFO] [launch.py:155:main] 0 NV_LIBNCCL_DEV_PACKAGE=libnccl-dev=2.25.1-1+cuda12.8
[2026-06-30 21:55:22,739] [INFO] [launch.py:155:main] 0 NV_LIBNCCL_DEV_PACKAGE_VERSION=2.25.1-1
[2026-06-30 21:55:22,739] [INFO] [launch.py:155:main] 0 NCCL_VERSION=2.25.1-1
[2026-06-30 21:55:22,739] [INFO] [launch.py:155:main] 0 NV_LIBNCCL_DEV_PACKAGE_NAME=libnccl-dev
[2026-06-30 21:55:22,739] [INFO] [launch.py:155:main] 0 NV_LIBNCCL_PACKAGE=libnccl2=2.25.1-1+cuda12.8
[2026-06-30 21:55:22,739] [INFO] [launch.py:155:main] 0 NV_LIBNCCL_PACKAGE_NAME=libnccl2
[2026-06-